#  NHS Workforce Hierarchical Forecast — Dashboard Export

In [16]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.evaluation import evaluate
from hierarchicalforecast.methods import BottomUp, MinTrace
from hierarchicalforecast.utils import aggregate
from statsforecast import StatsForecast
from statsforecast.models import Naive

## 1. 📦 Data Load and Tidy-up
- Loads all NHS monthly workforce records and standardizes columns for consistency.
- Filters for groups with **≥24 months** of data, ensuring robust and comparable forecasts for dashboarding.

In [17]:
base_dir = r"C:\Users\Destinee.HassanBien\Documents\Desys Work"
all_agg_file = os.path.join(base_dir, "combined_workforce_monthly_new.csv")
all_agg = pd.read_csv(all_agg_file, parse_dates=["month"])
all_agg.columns = all_agg.columns.str.lower().str.strip()

In [18]:
group_cols = ["service group", "staff group", "metric"]
count = all_agg.groupby(group_cols).size()
valid_groups = set(count[count >= 24].index)
all_agg_hier = all_agg.set_index(group_cols)
all_agg_hier = all_agg_hier.iloc[all_agg_hier.index.isin(valid_groups)].reset_index()

## 2. 🏗️ Build the NHS Workforce Hierarchy
Tip
- This step ensures every time series (“unique_id”) is recognizable at Trust, service, staff group, and metric levels—enabling proper aggregation and dashboard filters.

In [19]:
all_agg_hier["Total"] = "NHS"
spec = [
    ["Total"],
    ["Total", "service group"],
    ["Total", "service group", "staff group"],
    ["Total", "service group", "staff group", "metric"],
]
ys = all_agg_hier.rename(
    columns={
        "month": "ds",
        "count": "y",
        "service group": "service group",
        "staff group": "staff group",
        "metric": "metric",
    }
)
Y_df, S_df, tags = aggregate(ys, spec)
Y_df = Y_df.sort_values(["unique_id", "ds"])

## 3. ⏳ Set Extended Forecast Horizon to March 2027
- Extend the forecast so your dashboard always covers **the coming NHS financial year**.

In [20]:
# Find last observed date
observed_last = Y_df["ds"].max()
# Target horizon: from (observed_last + 1 month) up to March 2027
forecast_until = pd.Timestamp("2027-03-01")
n_months_forward = (forecast_until.year - observed_last.year) * 12 + (
    forecast_until.month - observed_last.month
)
horizon = max(n_months_forward, 12)

## 4. 🔮 Forecasting and Hierarchical Reconciliation
Why hierarchical forecasting?
- Ensures all group and roll-up forecasts “add up” for Trust-wide, service-line, or metric-level dashboards (what senior NHS workforce wants to see).

In [21]:
sf = StatsForecast(models=[Naive()], freq="MS", n_jobs=-1)
# Use ALL available data for most up to date forecast
Y_hat_df = sf.forecast(h=horizon, df=Y_df, fitted=True)
Y_fitted_df = sf.forecast_fitted_values()

In [22]:
reconcilers = [BottomUp(), MinTrace(method="ols"), MinTrace("mint_shrink")]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_df, Y_df=Y_fitted_df, S=S_df, tags=tags)

## 5. 📊 Prepare for Dashboard: Join Observed & All Forecasts
Note:
- Both observed (“historic”) and predicted (“future”) values are needed for meaningful dashboard time series line charts.
- Merge all methods and all groups/metrics into a **single long “tidy” table**.


In [23]:
obs_df = (
    Y_df[["unique_id", "ds", "y"]]
    .groupby(["unique_id", "ds"], as_index=False)
    .mean()
    .rename(columns={"y": "observed"})
)

In [ ]:
all_results = pd.merge(Y_rec_df, obs_df, on=["unique_id", "ds"], how="outer")

## 6. 🔗 Split unique_id for Easier Filtering in Dash
- By splitting each unique_id into “service_group”, “staff_group”, and “metric”, the dashboard can offer multi level drilldown filters and breakdowns in dropdown menus, panel selectors, or slicers.

In [25]:
split_cols = all_results["unique_id"].str.split("/", expand=True)
split_cols.columns = ["org", "service_group", "staff_group", "metric"]
for col in ["service_group", "staff_group", "metric"]:
    all_results[col] = split_cols[col]

## 7. 💾 Save Master Table for Dashboards
Export for Dash
- The exported .csv contains all dates, groups, metrics, and future forecast methods.
- Plug this file directly into **Plotly Dash, Power BI, Tableau, Excel**, or R Shiny dashboards for instant workforce time series analytics!

In [26]:
out_path = os.path.join(base_dir, "nhs_hierarchical_forescasts_long1.csv")
all_results.to_csv(out_path, index=False)
print(f"\nDashboard-ready results exported: {out_path}")


Dashboard-ready results exported: C:\Users\Destinee.HassanBien\Documents\Desys Work\nhs_hierarchical_forescasts_long1.csv


## ✅ Summarise for Dashboard Documentation
Key Points
- All-line, all-method, all-history+future time series, grouped by Trust hierarchy.
- Dashboard filtering is robust and fast since all identifiers are explicitly encoded as columns.
- This output is the **standard “data lens”** for any interactive NHS workforce analytics app.